# Long-Context Decode Benchmark: Split-KV vs SDPA on T4

Extends `triton-attention-kernels` to 32K-64K decode. Run on Kaggle T4 (GPU).

Produces a clean table: latency vs PyTorch SDPA across context lengths and split counts.

In [ ]:
!pip -q install triton torch --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || pip -q install triton torch
!git clone -q https://github.com/YuvrajSinghBhadoria2/triton-attention-kernels.git /tmp/tak 2>/dev/null || (cd /tmp/tak && git pull -q)
import sys; sys.path.insert(0, '/tmp/tak')
import torch, triton
print('torch', torch.__version__, 'triton', triton.__version__)
print('GPU', torch.cuda.get_device_name(0))

In [ ]:
from kernels.flash_decoding import flash_decoding

B, H, D = 1, 8, 64
DEVICE = 'cuda'

def make_inputs(n_ctx):
    q = torch.randn(B, H, 1, D, device=DEVICE, dtype=torch.float16)
    k = torch.randn(B, H, n_ctx, D, device=DEVICE, dtype=torch.float16)
    v = torch.randn(B, H, n_ctx, D, device=DEVICE, dtype=torch.float16)
    return q, k, v

def sdpa_decode(q, k, v):
    # q: [B,H,1,D] -> [B,H,1,D]; SDPA handles Q=1 efficiently
    return torch.nn.functional.scaled_dot_product_attention(q, k, v)

def bench(fn, warmup=10, iters=50):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters): fn()
    end.record(); torch.cuda.synchronize()
    return start.elapsed_time(end) / iters

def check(q, k, v, out):
    ref = sdpa_decode(q, k, v)
    return torch.allclose(out.float(), ref.float(), atol=0.02, rtol=0.02)

In [ ]:
CTX = [1024, 2048, 4096, 8192, 16384, 32768, 65536]
SPLITS = [1, 4, 8, 16, 32, 64]
rows = []
for n in CTX:
    q, k, v = make_inputs(n)
    sdpa_ms = bench(lambda: sdpa_decode(q, k, v))
    best_ms, best_s, ok = None, None, False
    for s in SPLITS:
        if s > n // 64:  # need at least one 64-token tile per split
            continue
        try:
            out = flash_decoding(q, k, v, num_splits=s)
            ms = bench(lambda: flash_decoding(q, k, v, num_splits=s))
            correct = check(q, k, v, out)
        except Exception as e:
            print(f'n={n} s={s} ERROR {type(e).__name__}: {e}')
            continue
        print(f'n={n:>6} splits={s:>3}  {ms:.3f} ms  sdpa={sdpa_ms:.3f} ms  ratio={ms/sdpa_ms:.2f}x  ok={correct}')
        if best_ms is None or ms < best_ms:
            best_ms, best_s, ok = ms, s, correct
    if best_ms is not None:
        rows.append((n, best_s, best_ms, sdpa_ms, best_ms/sdpa_ms, ok))
print('DONE')

In [ ]:
print('| Context | Best splits | Triton (ms) | SDPA (ms) | Triton/SDPA | Correct |')
print('|---:|---:|---:|---:|---:|:---:|')
for n, s, t, sd, r, ok in rows:
    print(f'| {n} | {s} | {t:.3f} | {sd:.3f} | {r:.2f}x | {ok} |')